# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.  
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules to keep in mind 💡
1. Every bar chart **must have a zero baseline** — no exceptions
2. Every chart **must have an insight title**, not a topic title
3. Aim for **professional quality** — clean background, readable font, no clutter
4. Horizontal bars for long category names

---


## Setup — Run this cell first


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Read the supplied World Happiness Report 2023 CSV.
# Supports the original course folder layout and the uploaded-file layout.
candidate_paths = [
    Path('../data/world_happiness_2023.csv'),
    Path('world_happiness_2023.csv'),
    Path('/mnt/data/world_happiness_2023.csv'),
]
csv_path = next((path for path in candidate_paths if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Could not find world_happiness_2023.csv. Place it beside this notebook "
        "or in ../data/."
    )

df = pd.read_csv(csv_path)
df.columns = [
    'Country', 'Region', 'Happiness_Score', 'GDP', 'Social_Support',
    'Life_Expectancy', 'Freedom', 'Generosity', 'Corruption'
]

print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(f"Loaded from: {csv_path}")
df.head()


Dataset: 63 countries, 9 columns
Loaded from: ../data/world_happiness_2023.csv


,Country,Region,Happiness_Score,GDP,Social_Support,Life_Expectancy,Freedom,Generosity,Corruption
0,Finland,Western Europe,7.804,10.775,0.954,71.9,0.949,0.142,0.179
1,Denmark,Western Europe,7.586,10.933,0.954,72.7,0.931,0.168,0.234
2,Iceland,Western Europe,7.525,10.878,0.983,72.5,0.961,0.260,0.150
3,Israel,Middle East and North Africa,7.473,10.527,0.916,72.4,0.903,0.149,0.826
4,Netherlands,Western Europe,7.464,11.015,0.939,72.4,0.879,0.240,0.296


In [2]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

---
## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.


In [3]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Compute mean happiness score by region, highest first.
region_avg = (
    df.groupby('Region', as_index=False)['Happiness_Score']
      .mean()
      .sort_values('Happiness_Score', ascending=False)
)

print(region_avg.to_string(index=False))

# Horizontal bars: reverse the data order so the highest region appears at the top.
fig1 = px.bar(
    region_avg.sort_values('Happiness_Score', ascending=True),
    x='Happiness_Score',
    y='Region',
    orientation='h',
    text=region_avg.sort_values('Happiness_Score', ascending=True)['Happiness_Score'].map(lambda x: f'{x:.2f}'),
    color='Happiness_Score',
    color_continuous_scale=['#d9e8f5', '#1f5a85'],
    title='Western Europe leads regional happiness, well ahead of lower-scoring regions',
    labels={'Happiness_Score': 'Average happiness score', 'Region': ''},
)

fig1.update_traces(textposition='outside', cliponaxis=False)
fig1.update_layout(
    template='plotly_white',
    title_x=0.02,
    title_font_size=20,
    xaxis=dict(range=[0, region_avg['Happiness_Score'].max() * 1.18],
               rangemode='tozero', showgrid=True, gridcolor='#e8edf2'),
    yaxis=dict(title='', tickfont=dict(size=12)),
    coloraxis_showscale=False,
    margin=dict(l=20, r=35, t=90, b=50),
    height=520,
)
fig1.show()


                      Region  Happiness_Score
              Western Europe         7.085533
       North America and ANZ         7.018250
  Central and Eastern Europe         6.338143
                   East Asia         5.966000
 Latin America and Caribbean         5.699000
              Southeast Asia         5.695250
Middle East and North Africa         4.943333
          Sub-Saharan Africa         4.064714
                  South Asia         3.618250


---
## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.


In [4]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8 happiest'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8 least happy'

combined = pd.concat([bottom8, top8], ignore_index=True).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
score_gap = top8['Happiness_Score'].mean() - bottom8['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")
print(f"Difference between the top-8 and bottom-8 average scores: {score_gap:.2f}")

# Keep a deliberate two-colour palette; use the global average as a reference line.
color_map = {'Top 8 happiest': '#218c74', 'Bottom 8 least happy': '#c45a50'}
fig2 = px.bar(
    combined,
    x='Happiness_Score',
    y='Country',
    orientation='h',
    color='Group',
    color_discrete_map=color_map,
    text=combined['Happiness_Score'].map(lambda x: f'{x:.2f}'),
    title=f'The happiest 8 countries average {score_gap:.2f} points above the least happy 8',
    labels={'Happiness_Score': 'Happiness score', 'Country': '', 'Group': 'Country group'},
    category_orders={'Country': combined['Country'].tolist()},
)

fig2.add_vline(
    x=global_avg, line_dash='dash', line_width=2, line_color='#34495e',
    annotation_text=f'Global average: {global_avg:.2f}',
    annotation_position='top',
)
fig2.update_traces(textposition='outside', cliponaxis=False)
fig2.update_layout(
    template='plotly_white',
    title_x=0.02,
    title_font_size=19,
    xaxis=dict(range=[0, combined['Happiness_Score'].max() * 1.2],
               rangemode='tozero', showgrid=True, gridcolor='#e8edf2'),
    yaxis=dict(title='', tickfont=dict(size=11)),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(l=20, r=30, t=115, b=50),
    height=650,
)
fig2.show()


Global average: 5.81
Difference between the top-8 and bottom-8 average scores: 4.16


---
## Done? Stretch Goal

If you finish both tasks with time to spare, try this:

**Task 3 (stretch):** Build a **grouped bar chart** comparing 2 sub-factors (e.g. `GDP_per_capita` and `Freedom`) across the 5 most populated regions. Use colour meaningfully and write an insight title.

Regions to include: `'Western Europe'`, `'Latin America'`, `'East Asia'`, `'Sub-Saharan Africa'`, `'South Asia'`


In [5]:
# Stretch goal — grouped bar chart
# Compare GDP per capita and freedom across the five named regions that are present
# in this dataset. (The source CSV has "Latin America and Caribbean", rather than
# the shortened "Latin America" label in the prompt.)

requested_regions = [
    'Western Europe',
    'Latin America and Caribbean',
    'East Asia',
    'Sub-Saharan Africa',
    'South Asia',
]
region_factors = (
    df[df['Region'].isin(requested_regions)]
      .groupby('Region', as_index=False)[['GDP', 'Freedom']]
      .mean()
)
# Put regions in the requested order, omitting none that exist in the source.
region_factors['Region'] = pd.Categorical(
    region_factors['Region'], categories=requested_regions, ordered=True
)
region_factors = region_factors.sort_values('Region')

factors_long = region_factors.melt(
    id_vars='Region',
    value_vars=['GDP', 'Freedom'],
    var_name='Factor',
    value_name='Mean value',
)
fig3 = px.bar(
    factors_long,
    x='Region',
    y='Mean value',
    color='Factor',
    barmode='group',
    color_discrete_map={'GDP': '#4267a9', 'Freedom': '#e09f3e'},
    title='GDP scores exceed freedom scores in every selected region',
    labels={'Region': '', 'Mean value': 'Average indicator value', 'Factor': 'Indicator'},
)
fig3.update_layout(
    template='plotly_white',
    title_x=0.02,
    title_font_size=19,
    yaxis=dict(rangemode='tozero', showgrid=True, gridcolor='#e8edf2'),
    xaxis=dict(tickangle=-15),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(l=20, r=25, t=100, b=80),
    height=480,
)
fig3.show()

print("Insight: compare the paired bars within each region; GDP and Freedom are different indicators and are shown on their original scales.")


Insight: compare the paired bars within each region; GDP and Freedom are different indicators and are shown on their original scales.
